In [ ]:
# === TRAINING PRESETS ===
# Dataset: 220,000 samples available

PRESET = 1

configs = {
    0: {  # Ultra-Fast Debug - 16k steps (~3 min) - test code changes only
        'TRAIN_DATA_SIZE': 32_768,
        'N_ENVS': 8,
        'N_STEPS': 1024,
        'BATCH_SIZE': 128,
        'NUM_ITERATIONS': 1,
    },
    1: {  # Full Training - 262k steps (~45 min) - production training
        'TRAIN_DATA_SIZE': 32_768,
        'N_ENVS': 8,
        'N_STEPS': 2048,
        'BATCH_SIZE': 64,
        'NUM_ITERATIONS': 4,
    },
    2: {  # Extended - 524k steps (~90 min)
        'TRAIN_DATA_SIZE': 131_072,
        'N_ENVS': 8,
        'N_STEPS': 2048,
        'BATCH_SIZE': 64,
        'NUM_ITERATIONS': 4,
    },
    3: {  # Max - 1.04M steps (~3 hours)
        'TRAIN_DATA_SIZE': 200_000,
        'N_ENVS': 8,
        'N_STEPS': 2048,
        'BATCH_SIZE': 64,
        'NUM_ITERATIONS': 4,
    },
}

# Load config
cfg = configs[PRESET]
TRAIN_DATA_SIZE = cfg['TRAIN_DATA_SIZE']
N_ENVS = cfg['N_ENVS']
N_STEPS = cfg['N_STEPS']
NUM_ITERATIONS = cfg['NUM_ITERATIONS']
BATCH_SIZE = cfg['BATCH_SIZE']

# Fixed params
LOOKBACK_WINDOW = 288
HIDDEN_DIM = 64
POLICY_LAYERS = [256, 128]
VALUE_LAYERS = [128, 64]

# Hyperparameters - TUNED FOR v28 REWARD (3:1 ratio, fixed rewards)
LEARNING_RATE_START = 1e-4      # REDUCED from 3e-4 (more stable learning)
LEARNING_RATE_DECAY = 0.0
N_EPOCHS = 4                     # REDUCED from 10 (less overfitting)
ENT_COEF = 0.5                   # INCREASED from 0.25 (MORE exploration!)
CLIP_RANGE_VF = 0.2
CLIP_RANGE = 0.2                 # INCREASED from 0.1 (allow bigger policy changes)
TARGET_KL = 0.05                 # REDUCED from 0.10 (more conservative updates)
VF_COEF = 0.5                    # REDUCED from 3.0 (balance policy and value)
MAX_GRAD_NORM = 0.5
USE_SDE = False

DATA_SYMBOL = 'BTCUSDT'
DATA_TIMEFRAME = '5m'
DATA_PATH = f'data/binance-{DATA_SYMBOL}-{DATA_TIMEFRAME}.pkl'
MODEL_SAVE_PATH = "trading_bot"
VECNORM_SAVE_PATH = "vecnormalize.pkl"

# Calculated
TOTAL_TIMESTEPS = TRAIN_DATA_SIZE * NUM_ITERATIONS

print(f"Preset {PRESET}: {TOTAL_TIMESTEPS:,} steps | {NUM_ITERATIONS} iters | {N_ENVS} envs | {TRAIN_DATA_SIZE:,} samples")
print(f"\n🎯 HYPERPARAMETERS TUNED FOR v28 REWARD (3:1 fixed rewards):")
print(f"   ENT_COEF: 0.5 (HIGH entropy = more exploration)")
print(f"   VF_COEF: 0.5 (balanced policy/value learning)")
print(f"   N_EPOCHS: 4 (less overfitting)")
print(f"   LEARNING_RATE: 1e-4 (stable updates)")
print(f"   CLIP_RANGE: 0.2 (allow bigger policy changes)")
print(f"   TARGET_KL: 0.05 (conservative KL divergence)")
print(f"\n   Expected: More LONG/SHORT actions, less CLOSE spam")
print(f"   Goal: Win rate >25% to be profitable (3:1 ratio)")


Preset 1: 131,072 steps | 4 iters | 8 envs | 32,768 samples

🎯 HYPERPARAMETERS TUNED FOR v28 REWARD (3:1 fixed rewards):
   ENT_COEF: 0.5 (HIGH entropy = more exploration)
   VF_COEF: 0.5 (balanced policy/value learning)
   N_EPOCHS: 4 (less overfitting)
   LEARNING_RATE: 1e-4 (stable updates)
   CLIP_RANGE: 0.2 (allow bigger policy changes)
   TARGET_KL: 0.05 (conservative KL divergence)

   Expected: More LONG/SHORT actions, less CLOSE spam
   Goal: Win rate >25% to be profitable (3:1 ratio)


In [ ]:
import warnings
warnings.filterwarnings('ignore', message='enable_nested_tensor is True')
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.monitor import Monitor
from stable_baselines3 import PPO
import torch
import torch.nn as nn
import pandas as pd
import time
from environments.simple_trading_env import SimpleTradingEnv
from environments.trading_hybrid_extractor import TradingHybridExtractor

# Load data
df = pd.read_pickle(DATA_PATH)
train_data = df.iloc[0:TRAIN_DATA_SIZE].reset_index(drop=True)
print(f"Loaded {len(df):,} rows | Training on {len(train_data):,} samples")

# Setup policy - HYBRID CNN+GNN Architecture (OPTIMIZED)
# CNN: Temporal patterns (fast, 288 it/s)
# GNN: S/R levels, channels, Elliott waves (OPTIMIZED - vectorized loops!)
policy_kwargs = dict(
    features_extractor_class=TradingHybridExtractor,
    features_extractor_kwargs=dict(hidden_dim=HIDDEN_DIM, use_gnn=True),  # ENABLED with optimizations
    net_arch=dict(pi=POLICY_LAYERS, vf=VALUE_LAYERS),
    activation_fn=torch.nn.ReLU,
    ortho_init=False,
)

# Create environments
vec_env = make_vec_env(
    lambda: Monitor(SimpleTradingEnv(train_data, device="cuda", lookback_window=LOOKBACK_WINDOW)),
    n_envs=N_ENVS
)

# Create model
model = PPO(
    "MultiInputPolicy",
    vec_env,
    device="cuda",
    learning_rate=lambda f: LEARNING_RATE_START * (1 - LEARNING_RATE_DECAY * f),
    n_steps=N_STEPS,
    batch_size=BATCH_SIZE,
    n_epochs=N_EPOCHS,
    gamma=0.995,
    gae_lambda=0.95,
    clip_range=CLIP_RANGE,
    ent_coef=ENT_COEF,
    vf_coef=VF_COEF,
    max_grad_norm=MAX_GRAD_NORM,
    target_kl=TARGET_KL,
    stats_window_size=288,
    policy_kwargs=policy_kwargs,
    clip_range_vf=CLIP_RANGE_VF,
    use_sde=USE_SDE,
    tensorboard_log="./tensorboard_logs/",
    verbose=2
)


try:
    # Train
    print(f"\nStarting training: {TOTAL_TIMESTEPS:,} steps...")
    torch.cuda.empty_cache()
    model.learn(total_timesteps=TOTAL_TIMESTEPS, progress_bar=True)
except KeyboardInterrupt:
    print("\n⏸ Training interrupted by user")
    
# Save
model.save(f"{MODEL_SAVE_PATH}_hybrid")
print(f"\n✓ Saved: {MODEL_SAVE_PATH}_hybrid")

print("\n✅ Training complete!")

Loaded 264,323 rows | Training on 32,768 samples
Using cuda device
Using cuda device

Starting training: 131,072 steps...
Logging to ./tensorboard_logs/PPO_404

Starting training: 131,072 steps...
Logging to ./tensorboard_logs/PPO_404


Output()

------------------------------
| time/              |       |
|    fps             | 65    |
|    iterations      | 1     |
|    time_elapsed    | 249   |
|    total_timesteps | 16384 |
------------------------------


------------------------------------------
| time/                   |              |
|    fps                  | 50           |
|    iterations           | 2            |
|    time_elapsed         | 644          |
|    total_timesteps      | 32768        |
| train/                  |              |
|    approx_kl            | 0.0003975474 |
|    clip_fraction        | 0            |
|    clip_range           | 0.2          |
|    clip_range_vf        | 0.2          |
|    entropy_loss         | -1.39        |
|    explained_variance   | -0.000481    |
|    learning_rate        | 0.0001       |
|    loss                 | -0.212       |
|    n_updates            | 4            |
|    policy_gradient_loss | -0.000295    |
|    value_loss           | 0.988        |
------------------------------------------


Account bankrupt!

...

Account bankrupt!

Account bankrupt!

Account bankrupt!

Account bankrupt!

------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 5.66e+03     |
|    ep_rew_mean          | 294          |
| time/                   |              |
|    fps                  | 44           |
|    iterations           | 3            |
|    time_elapsed         | 1102         |
|    total_timesteps      | 49152        |
| train/                  |              |
|    approx_kl            | 0.0030076436 |
|    clip_fraction        | 0            |
|    clip_range           | 0.2          |
|    clip_range_vf        | 0.2          |
|    entropy_loss         | -1.38        |
|    explained_variance   | -6.52e-05    |
|    learning_rate        | 0.0001       |
|    loss                 | -0.266       |
|    n_updates            | 8            |
|    policy_gradient_loss | -0.00194     |
|    value_loss           | 0.954        |
------------------------------------------
